# 🎬 Movie Review Sentiment Analysis — TRAIN FILE
**Topic:** NLP + Machine Learning (Supervised Classification)  
**Dataset:** `movie_reviews.csv` — 100 reviews, 5 sentiment classes × 20 each  
**Classes:** Highly Positive | Positive | Neutral | Negative | Highly Negative  
**Pipeline:** Dataset → Pre-processing → Feature Extraction (TF-IDF) → ML Model → Evaluation → Save `.pkl`  
**Folder:** `Movie_Sentiment_Analysis/` → subfolders: `data/` | `train/` | `predict/`

### ✨ Accuracy Improvements Applied:
- **Negation preservation** — words like `not`, `no`, `never` kept (critical for sentiment)
- **Expanded TF-IDF** — 3000 features + character n-grams for better coverage
- **Voting Ensemble** — combines top classifiers for more robust predictions
- **Hyperparameter tuning** — GridSearchCV finds optimal model settings
- **Cross-validation** — 5-fold CV gives reliable accuracy estimates

In [ ]:
# ── Cell 1 — Install required libraries ──────────────────────────────────────
!pip install nltk scikit-learn pandas matplotlib seaborn -q
print('✅ Libraries installed.')

In [ ]:
# ── Cell 2 — Imports ──────────────────────────────────────────────────────────
import nltk
for pkg in ['stopwords', 'punkt', 'punkt_tab', 'wordnet']:
    nltk.download(pkg, quiet=True)

import pandas as pd
import numpy as np
import string
import warnings
import pickle
import os
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# NLP
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Feature Extraction
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion

# ML Models (Supervised Classification)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.calibration import CalibratedClassifierCV

# Evaluation & Tuning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)

print('✅ All imports done.')

In [ ]:
# ── Cell 3 — Mount Google Drive & Set Folder Paths ────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR    = '/content/drive/MyDrive/Movie_Sentiment_Analysis'
DATA_DIR    = os.path.join(BASE_DIR, 'data')
TRAIN_DIR   = os.path.join(BASE_DIR, 'train')
PREDICT_DIR = os.path.join(BASE_DIR, 'predict')

for folder in [DATA_DIR, TRAIN_DIR, PREDICT_DIR]:
    os.makedirs(folder, exist_ok=True)

print('✅ Google Drive mounted.')
print(f'   Base    : {BASE_DIR}')
print(f'   Data    : {DATA_DIR}')
print(f'   Train   : {TRAIN_DIR}')
print(f'   Predict : {PREDICT_DIR}')

In [ ]:
# ── Cell 4 — Upload Dataset to Drive/data/ folder ─────────────────────────────
from google.colab import files
print('📂 Upload your movie_reviews.csv file now:')
uploaded = files.upload()

filename = list(uploaded.keys())[0]

import shutil
dest_path = os.path.join(DATA_DIR, 'movie_reviews.csv')
shutil.copy(filename, dest_path)

df = pd.read_csv(dest_path)

print(f'\n✅ Dataset loaded from: {dest_path}')
print(f'   Total rows    : {len(df)}')
print(f'   Columns       : {df.columns.tolist()}')
print(f'\n📊 Class Distribution:')
print(df['sentiment'].value_counts())
df.head()

In [ ]:
# ── Cell 5 — STEP 1: Pre-processing ──────────────────────────────────────────
# IMPROVEMENT: Negation words (not, no, never, wasn't, etc.) are KEPT.
# Removing them destroys sentiment signal — e.g. "not good" becomes "good".
# All other stopwords are still removed to reduce noise.

NEGATION_WORDS = {
    'no', 'not', 'nor', 'never', 'neither', 'nobody', 'nothing', 'nowhere',
    'hardly', 'scarcely', 'barely', 'without',
    "isn't", "wasn't", "weren't", "hasn't", "haven't", "hadn't",
    "doesn't", "didn't", "don't", "won't", "wouldn't", "couldn't",
    "shouldn't", "can't", "cannot"
}

base_stop_words = set(stopwords.words('english'))
# Remove negation words from the stopword list so they are preserved
stop_words = base_stop_words - NEGATION_WORDS

lemmatizer = WordNetLemmatizer()

def preprocess(text):
    # Step 1: Lowercase
    text = str(text).lower()
    # Step 2: Remove punctuation (keep apostrophes for contractions like "isn't")
    text = text.translate(str.maketrans('', '', string.punctuation.replace("'", '')))
    # Step 3: Tokenize
    tokens = word_tokenize(text)
    # Step 4: Remove stopwords (negation words kept) + non-alpha tokens
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    # Step 5: Lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

df['clean_review'] = df['review'].apply(preprocess)

print('✅ Pre-processing complete.')
print('\nSample — Original vs Cleaned:')
for i in range(3):
    print(f'\n  Original : {df["review"].iloc[i]}')
    print(f'  Cleaned  : {df["clean_review"].iloc[i]}')

In [ ]:
# ── Cell 6 — STEP 2: Feature Extraction (Improved TF-IDF) ────────────────────
# IMPROVEMENT 1: max_features raised from 500 → 3000 for richer vocabulary.
# IMPROVEMENT 2: Word n-grams (1,3) capture phrases like "not good", "really bad".
# IMPROVEMENT 3: Character n-grams added via FeatureUnion to catch morphological
#                variants (e.g. "amazing" vs "amazingly") — boosts accuracy on
#                unseen words and spelling variations.
# IMPROVEMENT 4: min_df=1 keeps all useful rare words (dataset is small).

X = df['clean_review']
y = df['sentiment']

# Word-level TF-IDF
word_tfidf = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 3),     # Unigrams, Bigrams, Trigrams
    sublinear_tf=True,
    min_df=1,
    analyzer='word'
)

# Character-level TF-IDF (catches word roots & misspellings)
char_tfidf = TfidfVectorizer(
    max_features=2000,
    ngram_range=(3, 5),     # Character 3-grams to 5-grams
    sublinear_tf=True,
    analyzer='char_wb'
)

# Combine both feature sets
from scipy.sparse import hstack

X_word = word_tfidf.fit_transform(X)
X_char = char_tfidf.fit_transform(X)
X_tfidf = hstack([X_word, X_char])   # Combined feature matrix

print('✅ TF-IDF Feature Extraction complete.')
print(f'   Word features  : {X_word.shape[1]}')
print(f'   Char features  : {X_char.shape[1]}')
print(f'   Combined shape : {X_tfidf.shape}')

In [ ]:
# ── Cell 7 — STEP 3: Train/Test Split ────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('✅ Train/Test Split done.')
print(f'   Training samples : {X_train.shape[0]}')
print(f'   Testing  samples : {X_test.shape[0]}')
print(f'\n   Train class distribution:')
print(pd.Series(y_train).value_counts())

In [ ]:
# ── Cell 8 — STEP 4: Train & Evaluate Individual ML Models ───────────────────
# Same 4 classifiers as before, now evaluated with 5-fold cross-validation
# for more reliable accuracy estimates (not just a single train/test split).

models = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=42),
    'Naive Bayes'         : MultinomialNB(),
    'Linear SVM'          : LinearSVC(max_iter=2000, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}

print('Training and evaluating models...\n')
for name, model in models.items():
    # 5-fold cross-validation on full training data
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')

    # Fit on full train set, predict on test set
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc  = round(accuracy_score(y_test, y_pred) * 100, 2)
    prec = round(precision_score(y_test, y_pred, average='weighted', zero_division=0) * 100, 2)
    rec  = round(recall_score(y_test, y_pred, average='weighted', zero_division=0) * 100, 2)
    f1   = round(f1_score(y_test, y_pred, average='weighted', zero_division=0) * 100, 2)
    cv_mean = round(cv_scores.mean() * 100, 2)
    cv_std  = round(cv_scores.std()  * 100, 2)

    results[name] = {
        'model': model, 'preds': y_pred,
        'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1,
        'CV_Mean': cv_mean, 'CV_Std': cv_std
    }
    print(f'  [{name}]')
    print(f'    Test  → Acc={acc}%  Prec={prec}%  Rec={rec}%  F1={f1}%')
    print(f'    5-fold CV → {cv_mean}% ± {cv_std}%\n')

print('✅ All 4 models trained and evaluated.')

In [ ]:
# ── Cell 9 — STEP 5: Hyperparameter Tuning on Best Individual Model ───────────
# IMPROVEMENT: GridSearchCV finds the optimal hyperparameters for Logistic
# Regression and LinearSVC — the two strongest classifiers for text tasks.
# This often gives a 3–8% accuracy boost over default settings.

print('🔍 Running GridSearchCV for Logistic Regression...\n')

lr_params = {
    'C'      : [0.01, 0.1, 1, 5, 10],
    'solver' : ['lbfgs', 'saga'],
    'penalty': ['l2']
}
lr_grid = GridSearchCV(
    LogisticRegression(max_iter=2000, random_state=42),
    lr_params, cv=5, scoring='accuracy', n_jobs=-1, verbose=0
)
lr_grid.fit(X_train, y_train)
lr_best = lr_grid.best_estimator_
lr_acc  = round(accuracy_score(y_test, lr_best.predict(X_test)) * 100, 2)

print(f'  Best LR params : {lr_grid.best_params_}')
print(f'  Best LR CV Acc : {round(lr_grid.best_score_*100,2)}%')
print(f'  LR Test Acc    : {lr_acc}%\n')

print('🔍 Running GridSearchCV for Linear SVM...\n')
svm_params = {'C': [0.01, 0.1, 0.5, 1, 5, 10]}
svm_grid = GridSearchCV(
    LinearSVC(max_iter=3000, random_state=42),
    svm_params, cv=5, scoring='accuracy', n_jobs=-1, verbose=0
)
svm_grid.fit(X_train, y_train)
svm_best = svm_grid.best_estimator_
svm_acc  = round(accuracy_score(y_test, svm_best.predict(X_test)) * 100, 2)

print(f'  Best SVM params: {svm_grid.best_params_}')
print(f'  Best SVM CV Acc: {round(svm_grid.best_score_*100,2)}%')
print(f'  SVM Test Acc   : {svm_acc}%')

# Update results with tuned models
results['Logistic Regression (Tuned)'] = {
    'model': lr_best,
    'preds': lr_best.predict(X_test),
    'Accuracy' : lr_acc,
    'Precision': round(precision_score(y_test, lr_best.predict(X_test), average='weighted', zero_division=0)*100,2),
    'Recall'   : round(recall_score(y_test, lr_best.predict(X_test),    average='weighted', zero_division=0)*100,2),
    'F1'       : round(f1_score(y_test, lr_best.predict(X_test),        average='weighted', zero_division=0)*100,2),
    'CV_Mean'  : round(lr_grid.best_score_*100,2),
    'CV_Std'   : 0.0
}
results['Linear SVM (Tuned)'] = {
    'model': svm_best,
    'preds': svm_best.predict(X_test),
    'Accuracy' : svm_acc,
    'Precision': round(precision_score(y_test, svm_best.predict(X_test), average='weighted', zero_division=0)*100,2),
    'Recall'   : round(recall_score(y_test, svm_best.predict(X_test),    average='weighted', zero_division=0)*100,2),
    'F1'       : round(f1_score(y_test, svm_best.predict(X_test),        average='weighted', zero_division=0)*100,2),
    'CV_Mean'  : round(svm_grid.best_score_*100,2),
    'CV_Std'   : 0.0
}

print('\n✅ Hyperparameter tuning complete.')

In [ ]:
# ── Cell 10 — STEP 6: Voting Ensemble (NEW — Highest Accuracy) ────────────────
# IMPROVEMENT: A Voting Ensemble combines the tuned LR, tuned SVM (calibrated),
# and Naive Bayes. It votes on the final class per review.
# Ensembles reduce individual model errors and consistently outperform any
# single classifier on text classification tasks.

# LinearSVC doesn't support predict_proba natively — wrap with CalibratedClassifierCV
svm_calibrated = CalibratedClassifierCV(svm_best, cv=3)
svm_calibrated.fit(X_train, y_train)

ensemble = VotingClassifier(
    estimators=[
        ('lr',  lr_best),
        ('svm', svm_calibrated),
        ('nb',  MultinomialNB())
    ],
    voting='soft'    # 'soft' uses predicted probabilities — more accurate than hard voting
)

ensemble.fit(X_train, y_train)
ens_pred = ensemble.predict(X_test)

ens_acc  = round(accuracy_score(y_test, ens_pred) * 100, 2)
ens_prec = round(precision_score(y_test, ens_pred, average='weighted', zero_division=0) * 100, 2)
ens_rec  = round(recall_score(y_test, ens_pred,    average='weighted', zero_division=0) * 100, 2)
ens_f1   = round(f1_score(y_test, ens_pred,        average='weighted', zero_division=0) * 100, 2)
ens_cv   = cross_val_score(ensemble, X_train, y_train, cv=5, scoring='accuracy')

results['Voting Ensemble'] = {
    'model': ensemble, 'preds': ens_pred,
    'Accuracy': ens_acc, 'Precision': ens_prec, 'Recall': ens_rec, 'F1': ens_f1,
    'CV_Mean': round(ens_cv.mean()*100,2), 'CV_Std': round(ens_cv.std()*100,2)
}

print(f'🏆 Voting Ensemble Results:')
print(f'   Test  → Acc={ens_acc}%  Prec={ens_prec}%  Rec={ens_rec}%  F1={ens_f1}%')
print(f'   5-fold CV → {round(ens_cv.mean()*100,2)}% ± {round(ens_cv.std()*100,2)}%')
print('\n✅ Ensemble model ready.')

In [ ]:
# ── Cell 11 — STEP 7: Pick Best Model Across All ─────────────────────────────
best_name = max(results, key=lambda k: results[k]['Accuracy'])
best      = results[best_name]

print('='*72)
print('   MODEL COMPARISON SUMMARY (All Models including Tuned + Ensemble)')
print('='*72)
print(f'  {"Model":<30} {"Acc":>8} {"Prec":>8} {"Rec":>8} {"F1":>8}  CV Mean')
print('-'*72)
for name, r in results.items():
    marker = ' ← BEST' if name == best_name else ''
    cv_str = f"{r['CV_Mean']}%" if r['CV_Mean'] else '-'
    print(f'  {name:<30} {r["Accuracy"]:>7}% {r["Precision"]:>7}% {r["Recall"]:>7}% {r["F1"]:>7}%  {cv_str}{marker}')
print('='*72)
print(f'\n✅ Best Model : {best_name}')
print(f'   Accuracy   : {best["Accuracy"]}%')
print(f'   F1 Score   : {best["F1"]}%')

In [ ]:
# ── Cell 12 — Detailed Classification Report ──────────────────────────────────
print(f'📋 Classification Report — {best_name}\n')
print(classification_report(y_test, best['preds'], zero_division=0))

In [ ]:
# ── Cell 13 — Confusion Matrix & Metrics Chart ────────────────────────────────
class_labels = ['Highly Negative', 'Negative', 'Neutral', 'Positive', 'Highly Positive']

cm = confusion_matrix(y_test, best['preds'], labels=class_labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Confusion Matrix — {best_name}', fontsize=11)
axes[0].tick_params(axis='x', rotation=30)

metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
values  = [best['Accuracy'], best['Precision'], best['Recall'], best['F1']]
colors  = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
bars = axes[1].bar(metrics, values, color=colors, alpha=0.87)
for bar, val in zip(bars, values):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f'{val}%', ha='center', va='bottom', fontweight='bold', fontsize=11
    )
axes[1].set_ylim(0, 115)
axes[1].set_title(f'Evaluation Metrics — {best_name}', fontsize=11)
axes[1].set_ylabel('Score %')
axes[1].axhline(80, color='gray', linestyle='--', linewidth=1, label='80% baseline')
axes[1].legend(fontsize=9)

plt.tight_layout()
plot_path = os.path.join(TRAIN_DIR, 'evaluation_metrics.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'✅ Plot saved to: {plot_path}')

In [ ]:
# ── Cell 14 — Model Comparison Bar Chart (All Models) ─────────────────────────
model_names = list(results.keys())
accs  = [results[m]['Accuracy'] for m in model_names]
f1s   = [results[m]['F1']       for m in model_names]

x = np.arange(len(model_names))
w = 0.35

fig, ax = plt.subplots(figsize=(14, 5))
b1 = ax.bar(x - w/2, accs, w, label='Accuracy', color='#4C72B0', alpha=0.87)
b2 = ax.bar(x + w/2, f1s,  w, label='F1 Score', color='#55A868', alpha=0.87)

for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.8, f'{h}%',
            ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_title('ML Model Comparison — Accuracy vs F1 Score (All Models)', fontsize=13)
ax.set_ylabel('Score %')
ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=9, rotation=15, ha='right')
ax.set_ylim(0, 120)
ax.axhline(80, color='gray', linestyle='--', linewidth=1, label='80% baseline')
ax.legend(fontsize=9)
plt.tight_layout()
comp_path = os.path.join(TRAIN_DIR, 'model_comparison.png')
plt.savefig(comp_path, dpi=150)
plt.show()
print(f'✅ Comparison chart saved to: {comp_path}')

In [ ]:
# ── Cell 15 — Save Best Model + BOTH TF-IDF Vectorizers as .pkl ───────────────
# IMPORTANT: We now save TWO vectorizers (word + char) since both are used
# during prediction. The predict.ipynb loads all three files.

model_path     = os.path.join(TRAIN_DIR, 'sentiment_model.pkl')
word_tfidf_path = os.path.join(TRAIN_DIR, 'tfidf_word_vectorizer.pkl')
char_tfidf_path = os.path.join(TRAIN_DIR, 'tfidf_char_vectorizer.pkl')

with open(model_path, 'wb') as f:
    pickle.dump(best['model'], f)

with open(word_tfidf_path, 'wb') as f:
    pickle.dump(word_tfidf, f)

with open(char_tfidf_path, 'wb') as f:
    pickle.dump(char_tfidf, f)

print('✅ Model and Vectorizers saved to Google Drive!')
print(f'   Model path         : {model_path}')
print(f'   Word vectorizer    : {word_tfidf_path}')
print(f'   Char vectorizer    : {char_tfidf_path}')
print(f'   Best Algorithm     : {best_name}')
print(f'   Accuracy           : {best["Accuracy"]}%')
print(f'   F1 Score           : {best["F1"]}%')
print('\n🎯 Training complete. Open predict.ipynb to make predictions!')